# Build a RAG step by step

Markdown-only coding guide. Add code cells beneath these steps as you implement them. Build one working RAG over the processed football-prediction papers, then test it with the ten questions in `docs/rag-build-log.md`. The four papers are enough for a prototype, even if some questions are unsupported by this small corpus.

First version: load existing chunks → embed them → save a local vector index → retrieve passages → generate a grounded answer with paper/page citations. No paper-summary pipeline or agent orchestration is needed to get this working.

## 1. Inspect the existing chunks

Use `data/processed/chunks.jsonl`; do not repeat PDF ingestion or chunking. Load a few records, inspect text and metadata, and count documents/chunks. The corpus has 162 chunks from four papers. Preserve filename, document ID, page, chunk ID, and source path for citations. Note extraction issues, but do not pause to repair every PDF artifact.

## 2. Make searchable documents

Convert each JSONL record to a LangChain `Document` (or your store's equivalent). Put chunk text in `page_content` and preserve provenance as metadata. Inspect a converted record to confirm citation fields survive.

## 3. Embed and persist the index

Choose one embedding model and one persistent local vector store. `langchain-openai` is already declared; if using OpenAI embeddings, read the key from the environment or local `.env`, never hard-code or commit it. Save the rebuildable index under `data/vectorstore/`.

A persistent vector-store integration is not currently declared, so choose one supported local integration and add its package if needed (for example, Chroma). Index all chunks and verify you can reload the index in a fresh session. Record the embedding model and store type.

## 4. Retrieve passages

Implement a function that takes a question and returns the top few similar chunks. Start with similarity search and a modest `k` (such as 4–6). Try some gold questions and display each result's rank, paper/file, page, and excerpt. Check relevance and whether cross-paper questions retrieve distinct papers when evidence exists. Do not add query rewriting, hybrid search, or reranking yet.

## 5. Generate a grounded, cited answer

Send the question and retrieved passages to a chat model. Instruct it to answer only from those passages, preserve study-specific qualifiers, cite key claims with paper and page, distinguish a feature being mentioned from evidence that it helped, and state when evidence is insufficient or studies are not comparable. Return the answer together with retrieved documents so you can inspect its evidence.

## 6. Run one complete question

Connect question → retrieval → context formatting → answer generation → answer plus sources. Start with: **Which features appear consistently useful across the studies?** Check whether the retrieved evidence supports a cross-paper conclusion. Repeated use of a feature does not by itself prove consistent predictive usefulness.

## 7. Evaluate with the ten gold questions

Run all ten questions from `docs/rag-build-log.md`. Save each question, retrieved paper/page/chunk references, answer, and a short review. Classify corpus coverage as **answerable**, **partially answerable**, or **unsupported**. Also review retrieval relevance, answer grounding/qualification, and citation accuracy.

Separate: relevant text exists but retrieval missed it (retrieval issue); relevant text was retrieved but misrepresented (generation issue); or the papers do not contain the needed evidence (corpus gap). A corpus gap is not automatically a RAG failure. Model comparisons may not support a universal ranking when datasets, targets, and metrics differ.

## 8. Improve one observed failure

Use the review notes to choose one change. If relevant passages are missed, try retrieval depth or chunking. If useful results are buried, consider a reranker later. If answers overclaim, improve the prompt or context formatting. If evidence is absent, record a corpus gap. Rerun the same questions and compare with the baseline. Synthetic questions are optional and can wait unless they help diagnose retrieval.

## 9. Move stable parts into the package

When the notebook flow works, move reusable indexing, retrieval, and generation functions into `src/rag_agent/` and import them here. Keep the notebook for experiments. A single retrieval-and-answer path is enough; add LangGraph only if the workflow later needs branching, retries, routing, or multiple retrieval steps.